In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import scipy as sp
import seaborn as sns

CROP_NAMES = {
    "apple": "Apple",
    "avocado": "Avocado",
    "blueberry": "Blueberry",
    "cherry": "Cherry",
    "citrus": "Citrus",
    "hops": "Hops",
    "kiwifruit": "Kiwifruit",
    "maize": "Maize",
    "manuka": "Manuka",
    "pinotnoir": "Pinot noir",
    "sauvignonblanc": "Sauvignon blanc",
    "wheat": "Wheat",
}

In [ ]:
def open_crop_data(crops):
    """Open crop suitability comparison data for the given crops."""
    data = {}
    for crop in crops:
        crop_path = Path().cwd() / "data" / f"{crop}_suitability_comparison.csv"
        data[crop] = pd.read_csv(crop_path)
    return data

In [ ]:
crops = ["apple", "avocado", "blueberry", "cherry", "kiwifruit", "pinotnoir", "sauvignonblanc"]
data = open_crop_data(crops)

In [ ]:
scatter_kws = {"alpha": 0.2, "edgecolor": "none", "color": "grey", "s": 15}
line_kws = {"color": "r", "lw": 1}
letter = ["a", "b", "c", "d", "e", "f", "g"]

fig, axs = plt.subplots(2, 4, figsize=(12, 6), sharey=True, sharex=True)
for i, crop in enumerate(crops):
    cname = CROP_NAMES[crop]
    dfcrop = data[crop]
    ax = axs[i // 4, i % 4]
    sns.regplot(
        x="NZLUSDB",
        y="DataSupermarket",
        data=dfcrop,
        ax=ax,
        marker=".",
        scatter_kws=scatter_kws,
        line_kws=line_kws,
        ci=None,
    )
    r, p = sp.stats.pearsonr(dfcrop["NZLUSDB"], dfcrop["DataSupermarket"])
    ax.set_title(f"({letter[i]}) {cname} | R$^2$={r**2:.2f}", fontsize=11, color="k", pad=5, loc="left")
    ax.set_xlabel("")
    ax.set_ylabel("")
axs[1, 3].axis("off")
fig.text(0.5, 0.01, "Suitability (NZLUSDB)", ha="center", fontsize=11)
fig.text(0.01, 0.5, "Suitability (DataSupermarket)", va="center", rotation="vertical", fontsize=11)
plt.subplots_adjust(left=0.06, right=0.95, top=0.95, bottom=0.08, wspace=0.1, hspace=0.15)
plt.savefig(Path().cwd() / "suitability_score_comparison.png", dpi=300)

In [ ]:
cats = {1: "WS", 2: "S", 3: "MS", 4: "US"}
order = list(cats.values())
data = open_crop_data(["citrus", "hops", "maize", "manuka", "wheat"])

coeff_r = {}
for c in data:
    r, p = sp.stats.pearsonr(data[c]["NZLUSDB"], data[c]["DataSupermarket"])
    coeff_r[c] = r**2
    df = data[c].replace(cats).value_counts().reset_index()
    df = df.pivot(index="DataSupermarket", columns="NZLUSDB", values="count")
    data[c] = df.reindex(index=order, columns=order)

In [ ]:
from matplotlib.colors import LogNorm

In [ ]:
letters = ["a", "b", "c", "d", "e"]
norm = LogNorm(vmin=10, vmax=10000)

fig, axs = plt.subplots(2, 3, figsize=(12, 6), sharey=True, sharex=True)
for i, crop in enumerate(data):
    cname = CROP_NAMES[crop]
    dfcrop = data[crop]
    ax = axs[i // 3, i % 3]
    sns.heatmap(
        dfcrop,
        annot=True,
        fmt=".0f",
        cmap="YlGnBu",
        norm=norm,
        cbar_kws={"label": "Count"},
        linewidths=0.5,
        linecolor="white",
        ax=ax,
    )
    ax.set_title(f"({letters[i]}) {cname} | R$^2$={coeff_r[crop]:.2f}", fontsize=11, color="k", pad=5, loc="left")
    ax.set_xlabel("")
    ax.set_ylabel("")
axs[1, 2].axis("off")
fig.text(0.5, 0.01, "Suitability (NZLUSDB)", ha="center", fontsize=11)
fig.text(0.01, 0.5, "Suitability (DataSupermarket)", va="center", rotation="vertical", fontsize=11)
plt.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.08, wspace=0.15, hspace=0.15)
plt.savefig(Path().cwd() / "suitability_category_comparison.png", dpi=300)